In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Klasterovanje multimodalnih senzorskih podataka na MEx skupu podataka

Ovaj projekat se bavi klasterovanjem multimodalnih senzorskih podataka prikupljenih tokom izvođenja fizioterapeutskih vežbi. Cilj je grupisati sesije vežbanja na osnovu senzorskih signala i evaluirati da li se dobijeni klasteri poklapaju sa stvarnim tipovima vežbi.

## O MEx skupu podataka (MEx Multi-modal Exercise Dataset)

MEx skup podataka sadrži multimodalne zapise **7 različitih fizioterapeutskih vežbi** koje je izvodilo **30 subjekata**. Svaka vežba je izvođena u trajanju od maksimalno 60 sekundi.

### Korišćeni senzori
1.  **Axivity AX3 3-Axis Logging Accelerometer (na butini - `act`):** 
2.  **Axivity AX3 3-Axis Logging Accelerometer (na ručnom zglobu - `acw`):**
3.  **Sensing Tex Pressure Mat (prostirka pritiska - `pm`):**
4.  **Obbrec Astra Depth Camera (dubinska kamera - `dc`):**

###  Postavka senzora tokom vežbanja
Subjekti su izvodili sve vežbe u ležećem položaju na prostirci pritiska. Dubinska kamera je bila postavljena iznad njih (snimajući iz ptičje perspektive), okrenuta nadole i poravnata sa gornjom ivicom prostirke pritiska i ramenima subjekta (tako da lice subjekta ne bude snimljeno radi očuvanja privatnosti).



In [ ]:
import os
import glob

data_dir = "mex/data"
subjects = [f"{i:02d}" for i in range(1, 31)]

meta = []

for s in subjects:
    accelometer_path = os.path.join(data_dir,"act",s)
    if not os.path.exists(accelometer_path):
        continue

    accelometer_files = sorted(glob.glob(os.path.join(accelometer_path, "*.csv")))
    for af in accelometer_files:
        fname = os.path.basename(af)
        parts = fname.split("_")
        ex = int(parts[0])
        trial = int(parts[2].split(".")[0])

        meta.append((s, ex, trial))


print(f"Ukupno pronađeno sesija: {len(meta)}")
print("Sesije prvog ispitanika (Subject, Exercise, Trial):")
for item in meta[:8]:
    print(item)

Ukupno pronađeno sesija: 239
Sesije prvog ispitanika (Subject, Exercise, Trial):
('01', 1, 1)
('01', 2, 1)
('01', 3, 1)
('01', 4, 1)
('01', 4, 2)
('01', 5, 1)
('01', 6, 1)
('01', 7, 1)


U ovom koraku pretražujemo fajl-sistem i mapiramo sve dostupne sesije snimanja analizirajući nazive CSV fajlova na putanji `mex/data/act`. 

Kao rezultat dobijamo listu `meta` koja sadrži torke u formatu `(Subject, Exercise, Trial)` za ukupno **239 uspešno snimljenih sesija**. Svaki ispitanik je izveo po 8 sesija (7 vežbi, pri čemu se vežba 4 izvodi na dve strane tela), osim ispitanika 22 koji je uradio 7 sesija (vežbu 4 je odradio na samo jednoj strani).

**Značenje metapodataka:**
*   **Subject (Ispitanik):** Jedinstveni identifikator osobe (od `'01'` do `'30'`).
*   **Exercise (Vežba):** Tip fizioterapeutske vežbe (od `1` do `7`). Ove oznake će nam kasnije služiti kao stvarna klasa za evaluaciju uspešnosti algoritama klasterovanja.
*   **Trial (Pokušaj/Strana):** Oznaka pokušaja ili strane tela na kojoj se vežba izvodila. Za vežbe koje se izvode asimetrično (vežba 4), imamo dva pokušaja (`1` i `2`), dok ostale vežbe imaju samo po jedan pokušaj (`1`).

**Svrha ovog koraka:**
Ova lista nam služi kao indeks (mapa) celog skupa podataka. Na osnovu nje možemo sinhronizovano i bezbedno učitati fajlove za preostala tri senzora (akcelerometar na ručnom zglobu, prostirka pritiska i dubinska kamera) za svaku pojedinačnu sesiju.


In [10]:
data = {}

for s, ex, trial in meta:
    act_path = os.path.join(data_dir, "act", s, f"{ex:02d}_act_{trial}.csv")
    acw_path = os.path.join(data_dir, "acw", s, f"{ex:02d}_acw_{trial}.csv")
    pm_path = os.path.join(data_dir, "pm_1.0_1.0", s, f"{ex:02d}_pm_{trial}.csv")
    dc_path = os.path.join(data_dir, "dc_0.05_0.05", s, f"{ex:02d}_dc_{trial}.csv")
    
    data[(s, ex, trial)] = {
        "act": pd.read_csv(act_path, header=None),
        "acw": pd.read_csv(acw_path, header=None),
        "pm": pd.read_csv(pm_path, header=None),
        "dc": pd.read_csv(dc_path, header=None)
    }

print(f"Podaci učitani. Učitano {len(data)} sesija")


Podaci učitani. Učitano 239 sesija
